In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt




In [9]:
data = pd.read_csv('data.csv')

columns_to_keep = ['age', 'gender','department', 'billing_amount', 'follow_up_required']
df = data[columns_to_keep]
df = pd.get_dummies(df, columns=["department"], prefix="", prefix_sep="", dtype=int)

print(df.head())



   age  gender billing_amount follow_up_required  Cardiology  General  \
0   76  female         £425.8                  1           1        0   
1   69       F        €344.26                  Y           0        0   
2   79       M        €203.34                  Y           0        0   
3   47       F        Rs85.76                 No           0        1   
4   45     NaN         $84.44                  1           0        1   

   Neurology  Orthopedics  
0          0            0  
1          0            1  
2          1            0  
3          0            0  
4          0            0  


In [10]:
gender_mapping = {
    'male': 1, 'Male': 1, 'M': 1, 'm': 1, '1': 1, 1.0: 1,
    'female': 0, 'Female': 0, 'F': 0, 'f': 0, '0': 0, 0.0: 0
}

follow_up_mapping = {
    'yes': 1, 'Yes': 1, 'y': 1, 'Y': 1, '1': 1, 1.0: 1,
    'no': 0, 'No': 0, 'N': 0, 'n': 0, '0': 0, 0.0: 0
}

df['gender'] = df['gender'].map(gender_mapping)
df['follow_up_required'] = df['follow_up_required'].map(follow_up_mapping)

df['billing_amount'] = df['billing_amount'].astype(str)
df['billing_amount'] = df['billing_amount'].str.replace(r'[£€$]', '', regex=True)
df['billing_amount'] = df['billing_amount'].str.replace('Rs', '', regex=False)
df['billing_amount'] = df['billing_amount'].str.replace('nan', '', regex=False)
df['billing_amount'] = pd.to_numeric(df['billing_amount'], errors='coerce')
df['billing_amount'] = df['billing_amount'].fillna(df['billing_amount'].median())

gender_mode = df['gender'].mode()[0]
follow_up_mode = df['follow_up_required'].mode()[0]

df['gender'] = df['gender'].fillna(gender_mode)
df['follow_up_required'] = df['follow_up_required'].fillna(follow_up_mode)

df['gender'] = df['gender'].astype(int)
df['follow_up_required'] = df['follow_up_required'].astype(int)

print(df['gender'].value_counts())
print(df['follow_up_required'].value_counts())
print(df.head())
print(df.columns.tolist())


gender
0    527
1    473
Name: count, dtype: int64
follow_up_required
1    514
0    486
Name: count, dtype: int64
   age  gender  billing_amount  follow_up_required  Cardiology  General  \
0   76       0          425.80                   1           1        0   
1   69       0          344.26                   1           0        0   
2   79       1          203.34                   1           0        0   
3   47       0           85.76                   0           0        1   
4   45       0           84.44                   1           0        1   

   Neurology  Orthopedics  
0          0            0  
1          0            1  
2          1            0  
3          0            0  
4          0            0  
['age', 'gender', 'billing_amount', 'follow_up_required', 'Cardiology', 'General', 'Neurology', 'Orthopedics']


In [11]:

X = df[['age', 'gender', 'billing_amount', 'Cardiology', 'General', 'Neurology', 'Orthopedics']].values
Y = df['follow_up_required'].values
X[:, 0] = (X[:, 0] - X[:, 0].mean()) / X[:, 0].std()
X[:, 2] = (X[:, 2] - X[:, 2].mean()) / X[:, 2].std()

print('starting........................')
X_train = X[:int(0.8 * len(X))]
Y_train = Y[:int(0.8 * len(Y))]

X_test = X[int(0.8 * len(X)):]
Y_test = Y[int(0.8 * len(Y)):]

W = np.zeros(X_train.shape[1])
b = np.random.rand(1)

def predict(W, X, b):
    return np.dot(X, W) + b

Z = 1 / (1 + np.exp(-predict(W, X_train, b)))

starting........................


In [12]:
def compute_cost(Z, Y_train):
    m = len(Y_train)
    cost = -1 / m * np.sum(Y_train * np.log(Z) + (1 - Y_train) * np.log(1 - Z))
    return cost

print(compute_cost(Z, Y_train))

0.6925270856743813


In [13]:
def compute_gradient(Z, X_train, Y_train):
    m = len(Y_train)
# z = [0.5, 0.7, 0.2, 0.9, 0.1]
# Y_train = [1, 0, 1, 0, 1]
    dw = 0
    db = 0
# [20, 1, 500.0, 0, 1, 0, 0]
    for i in range(m):
        error = Z[i] - Y_train[i]

        dw += error * X_train[i]
        db += error

    dw /= m
    db /= m

    return dw, db

In [14]:
def gradient_descent(X_train, Y_train, W, b, learning_rate, iterations):
    costs = []  # ✅ track cost every iteration
    
    for i in range(iterations):
        Z = 1 / (1 + np.exp(-predict(W, X_train, b)))
        
        dW, db = compute_gradient(Z, X_train, Y_train)
        W = W - learning_rate * dW
        b = b - learning_rate * db
        
        costs.append(compute_cost(Z, Y_train))  # ✅ save cost
    
    return W, b, costs  # ✅ return costs too


# Train
W, b, costs = gradient_descent(X_train, Y_train, W, b,
                                learning_rate=0.001,
                                iterations=1000)

Z_train = 1 / (1 + np.exp(-predict(W, X_train, b)))
Z_test  = 1 / (1 + np.exp(-predict(W, X_test, b)))

predictions = (Z_test >= 0.1).astype(int)
correct  = np.sum(predictions == Y_test)
accuracy = (correct / len(Y_test)) * 100
print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 49.00%
